In [ ]:
from pathlib import Path
from config import LOCAL_VOLUME, PROJECT_ROOT, get_git_commit

import modal

app = modal.App("LLM-training")

container = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_sync(uv_project_dir=PROJECT_ROOT)
    .add_local_python_source("transformer")
)

# same Volume name as etl.ipynb -- training reads the data/tokenizers ETL produced
volume = modal.Volume.from_name("LLM-pretraining", create_if_missing=True)

In [ ]:
@app.cls(
    image=container,
    volumes={"/storage": volume},
    # cpu=1.0,
    gpu="T4",
    timeout=86400,
    secrets=[modal.Secret.from_name("wandb-secret")],
)
class Trainer:
    """Training jobs for the pretraining pipeline. Reads data/tokenizers that
    etl.ipynb's ETL stages already committed to the Volume -- this class only
    trains, it doesn't prepare data.

    Layout (every path relative to VOLUME, same convention as ETL):
        data/{dataset_name}/bin/{tokenizer_uid}/{split}.bin   input  (from ETL)
        tokenizers/{tokenizer_uid}/...                        input  (from ETL)
        runs/{run_dir}/...                                    output (this class)

    `remote` drives everything location-specific: VOLUME (LOCAL_VOLUME vs the
    mount) and the commit/reload calls (Volume-API ops that only mean anything
    remotely -- locally we read/write disk directly).

    NOTE these are properties, not class attributes -- same reason as ETL: a
    class attribute is computed once at class-definition time (on the laptop,
    where is_local() is True), and that frozen value ships to the container
    as-is. A property re-evaluates is_local() at access time, on whichever
    side is actually running.
    """

    @property
    def remote(self) -> bool:
        return not modal.is_local()

    @property
    def VOLUME(self) -> Path:
        return Path("/storage") if self.remote else LOCAL_VOLUME

    @modal.method()
    def train(self, config_or_run_dir: dict | str, wandb_kwargs: dict | None = None):
        """Start a brand-new run (pass a config dict) or resume an existing one
        (pass its run_dir, relative to VOLUME) -- mirrors
        transformer.util.run_training's contract exactly, just dispatched
        through Modal instead of called directly.

        wandb_kwargs is passed straight through to run_training -- project,
        tags, entity, etc. are the caller's choice, not this class's; pass
        None to disable wandb entirely (e.g. for a local run with no
        wandb-secret env var to authenticate with)."""
        from transformer import run_training
        run_training(config_or_run_dir, self.VOLUME, wandb_kwargs=wandb_kwargs)


    @modal.method()
    def evaluate(self, run_dir: str):
        """Run eval/inference against a checkpoint under runs/{run_dir}/."""
        ...


In [ ]:
# --- og config: template for a sweep ---
import copy
from transformer import TransformerLM
from transformer.optimizer import lr_cosine_schedule
import torch

dataset_name = 'tinystories'
tokenizer_uid = 'tinystories-bpe-5k'
total_tokens_processed = 32768000  # batch_size x total_steps x context_length, held constant across the sweep

og_config = {
    "description": "tinystories_",
    "git_commit": get_git_commit(True),
    "seed": 0,
    "model_class": TransformerLM,
    "model_params": {
        "vocab_size": 5000,
        "context_length": 256,
        "num_layers": 4,
        "d_model": 512,
        "d_ff": 1344,
        "num_heads": 16,
        "rope_theta": 10000,
        "device": "cuda",
        "dtype": None,  # TODO: track this down - does this control all machine precision downstream?
    },
    "optimizer_class": torch.optim.AdamW,
    "optimizer_params": {
        "lr": 0.001,
        "betas": (0.9, 0.999),
        "weight_decay": 0.1,
        "eps": 1e-8,
    },
    "lr_schedule_fn": lr_cosine_schedule,
    "lr_schedule_params": {
        "max_learning_rate": 0.001,
        "min_learning_rate": 0.0001,
        "warmup_iters": 30,
        "cosine_cycle_iters": 1000,
    },
    "training": {
        "train_path": f"data/{dataset_name}/bin/{tokenizer_uid}/train.bin",
        "valid_path": f"data/{dataset_name}/bin/{tokenizer_uid}/valid.bin",
        "val_every": 10,
        "save_every": 500,
        "gpu_check_every": 50,
    },
}

# base kwargs shared by every run in the sweep -- per-run tags get appended below
wandb_kwargs_template = {
    "project": "llm-pretraining",
    "tags": [dataset_name, tokenizer_uid],
}

# --- sweep: deep-copy og_config, override entries, spawn each as its own detached run ---
# swap this list (and the two lines that use `batch_size` below) for whatever
# you actually want to sweep over.
batch_sizes = [32, 64, 128]

function_calls = []
with modal.enable_output():
    with app.run(detach=True):
        trainer = Trainer()
        for batch_size in batch_sizes:
            config = copy.deepcopy(og_config)  # independent nested dicts -- see note above
            config["description"] = f"{og_config['description']}_bs{batch_size}"
            config["training"]["batch_size"] = batch_size
            config["training"]["total_iterations"] = total_tokens_processed // (
                batch_size * config["model_params"]["context_length"]
            )

            wandb_kwargs = {**wandb_kwargs_template, "tags": [*wandb_kwargs_template["tags"], f"bs-{batch_size}"]}

            call = trainer.train.spawn(config, wandb_kwargs)  # non-blocking -- returns immediately
            function_calls.append(call)
            print(f"spawned batch_size={batch_size}: {call.object_id}")

# function_calls holds each run's FunctionCall handle. detach=True means these
# keep running on Modal even after this cell (and the notebook kernel) exits --
# nothing here blocks waiting for them. To check on one later, from any
# process: modal.FunctionCall.from_id(object_id).get() (blocks until done) or
# wrap in try/except modal.exception.OutputExpiredError to just poll.